# ReliabilityNet – Kaggle Model Server + Gradio UI

**Before you run:** in the right sidebar open **Session options** and set
- **Internet → On** (needs a phone-verified Kaggle account)
- **Accelerator → GPU** (optional, much faster)

Optional Kaggle Secrets (**Add-ons → Secrets**, tick each for this notebook):
- `NGROK_TOKEN` – public URL for the Flask API (free at https://dashboard.ngrok.com/)
- `HF_TOKEN` – only if the Hugging Face repo is private
- `ML_API_KEY` – require an `X-ML-Key` header on the API

This notebook:
1. Checks internet and installs missing packages (Flask, flask-cors, pyngrok, Gradio, …)
2. Downloads the three model artefacts from `rethes/ReliabilityNet`
3. Starts the Flask API as a background process on port 5001 (log: `/kaggle/working/flask.log`, view with `show_flask_log()`)
4. Tunnels it to the internet via ngrok at the fixed address **https://jockstrap-track-paradox.ngrok-free.dev** (skipped if no `NGROK_TOKEN`)
5. Launches a Gradio UI (new `gradio.live` link each run) for `/predict/sentiment`, `/predict/clickbait` and `/predict/reliability`


In [1]:
# ── 1. Pre-flight check + install dependencies ───────────────────────────────
# This notebook needs internet (pip, Hugging Face, ngrok, Gradio share link).
# Turn it on: right sidebar → Session options → Internet → On
# (requires a phone-verified Kaggle account), then restart the session.
import importlib
import importlib.util
import socket
import subprocess
import sys

def has_internet(host="huggingface.co"):
    try:
        socket.gethostbyname(host)
        return True
    except OSError:
        return False

if not has_internet():
    raise RuntimeError(
        "❌ No internet in this Kaggle session.\n"
        "   Right sidebar → Session options → Internet → On "
        "(needs a phone-verified account), then restart the session and Run All."
    )
print("✅ Internet is on")

# pip name → import name. Only missing packages are installed, so Kaggle's
# pre-installed versions (torch, transformers, gradio, …) are left alone.
PACKAGES = {
    "flask": "flask",
    "flask-cors": "flask_cors",
    "pyngrok": "pyngrok",
    "gradio": "gradio",
    "joblib": "joblib",
    "scipy": "scipy",
    "huggingface_hub": "huggingface_hub",
}

missing = [pip for pip, mod in PACKAGES.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Installing:", ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=False)
    importlib.invalidate_caches()

still_missing = [pip for pip, mod in PACKAGES.items() if importlib.util.find_spec(mod) is None]
if still_missing:
    raise RuntimeError(f"❌ These packages failed to install: {still_missing}")
print("✅ All packages available")

✅ Internet is on
Installing: flask-cors, pyngrok
✅ All packages available


In [ ]:
# ── 2. Download model weights from Hugging Face ──────────────────────────────
# Public repo: https://huggingface.co/rethes/ReliabilityNet
# snapshot_download caches files, so re-runs reuse the local copy.
import os
from huggingface_hub import snapshot_download

def get_secret(name):
    """Read a Kaggle Secret (Add-ons → Secrets). Kaggle Secrets are NOT
    environment variables, so os.environ alone never sees them."""
    value = os.environ.get(name)
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

HF_REPO_ID  = "rethes/ReliabilityNet"
MODEL_CACHE = "/kaggle/working/reliabilitynet_models"

MODEL_DIRS = [
    "roberta_sentiment_model",
    "roberta_large_clickbait_model",
    "meta_logistic_regression_model",
]

# Optional: only needed if the HF repo is private (Kaggle Secret named HF_TOKEN)
HF_TOKEN = get_secret("HF_TOKEN")
print("✅ HF_TOKEN found — private repo access enabled." if HF_TOKEN
      else "No HF_TOKEN set — using public access.")

print(f"Downloading models from https://huggingface.co/{HF_REPO_ID} ...")
repo_local_path = snapshot_download(
    repo_id        = HF_REPO_ID,
    local_dir      = MODEL_CACHE,
    allow_patterns = [f"{d}/**" for d in MODEL_DIRS],
    token          = HF_TOKEN,
)
print(f"✅ Models downloaded to: {repo_local_path}")

# Sanity-check: each directory must exist and contain files
problems = []
for d in MODEL_DIRS:
    p = os.path.join(MODEL_CACHE, d)
    files = os.listdir(p) if os.path.isdir(p) else []
    print(f"{'✅' if files else '❌ MISSING or EMPTY'}  {p}  ({len(files)} files)")
    if not files:
        problems.append(d)
if problems:
    raise RuntimeError(f"❌ Model folders missing or empty: {problems}")

No HF_TOKEN set — using public access.


In [ ]:
# ── 3. Set model paths ────────────────────────────────────────────────────────
SENTIMENT_MODEL_PATH = os.path.join(MODEL_CACHE, "roberta_sentiment_model")
CLICKBAIT_MODEL_PATH = os.path.join(MODEL_CACHE, "roberta_large_clickbait_model")
META_MODEL_PATH      = os.path.join(MODEL_CACHE, "meta_logistic_regression_model")

print("Model paths set:")
for p in [SENTIMENT_MODEL_PATH, CLICKBAIT_MODEL_PATH, META_MODEL_PATH]:
    print(f"  {'✅' if os.path.isdir(p) else '❌ MISSING'}  {p}")

In [ ]:
# ── 4. Inject model paths into the environment so model_server.py picks them up
os.environ["SENTIMENT_MODEL_PATH"] = SENTIMENT_MODEL_PATH
os.environ["CLICKBAIT_MODEL_PATH"] = CLICKBAIT_MODEL_PATH
os.environ["META_MODEL_PATH"]      = META_MODEL_PATH
os.environ["PORT"]                 = "5001"

# Optional: protect the public API with a key (Kaggle Secret named ML_API_KEY)
_ml_key = get_secret("ML_API_KEY")
if _ml_key:
    os.environ["ML_API_KEY"] = _ml_key
    print("✅ ML_API_KEY set — API requires the X-ML-Key header.")

# Allow connections from Gradio's internal proxy and ngrok
os.environ["ALLOWED_ORIGINS"] = "*"
print("✅ Environment set")

In [ ]:
# ── 5. Write model_server.py to disk ─────────────────────────────────────────
# Embedded so the notebook is self-contained. Raw string: written exactly as shown.

MODEL_SERVER_CODE = r'''
"""
ReliabilityNet – Flask model server.
Loads the saved sentiment, clickbait, and reliability models from disk,
receives article text, runs inference, and returns structured JSON.
"""

import hmac
import os
import sys
import torch
import torch.nn.functional as F
import joblib
import numpy as np
from scipy.sparse import hstack, csr_matrix
from flask import Flask, request, jsonify
from flask_cors import CORS
from transformers import AutoTokenizer, AutoModelForSequenceClassification

SENTIMENT_MODEL_PATH = os.environ.get("SENTIMENT_MODEL_PATH", "./roberta_sentiment_model")
CLICKBAIT_MODEL_PATH = os.environ.get("CLICKBAIT_MODEL_PATH", "./roberta_large_clickbait_model")
META_MODEL_PATH      = os.environ.get("META_MODEL_PATH",      "./meta_logistic_regression_model")
PORT         = int(os.environ.get("PORT", 5001))
ML_API_KEY   = os.environ.get("ML_API_KEY") or None
MAX_SENTIMENT_BODY_CHARS = 500
ALLOWED_ORIGINS = [
    origin.strip()
    for origin in (os.environ.get("ALLOWED_ORIGINS") or "*").split(",")
    if origin.strip()
]

# Use the Kaggle GPU when one is attached (Session options → Accelerator)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

if ML_API_KEY is None:
    print("WARNING: ML_API_KEY is not set; API key authentication is disabled.")

def to_device(batch):
    return {k: v.to(DEVICE) for k, v in batch.items()}

print(f"Loading RoBERTa Sentiment model from: {SENTIMENT_MODEL_PATH} ...")
try:
    sentiment_tokenizer = AutoTokenizer.from_pretrained(SENTIMENT_MODEL_PATH)
    sentiment_model = AutoModelForSequenceClassification.from_pretrained(SENTIMENT_MODEL_PATH)
    sentiment_model.to(DEVICE).eval()
    print("Sentiment model loaded.")
except Exception as e:
    print(f"Error loading sentiment model: {e}")
    sys.exit(1)

print(f"Loading RoBERTa Large Clickbait model from: {CLICKBAIT_MODEL_PATH} ...")
try:
    clickbait_tokenizer = AutoTokenizer.from_pretrained(CLICKBAIT_MODEL_PATH)
    clickbait_model = AutoModelForSequenceClassification.from_pretrained(CLICKBAIT_MODEL_PATH)
    clickbait_model.to(DEVICE).eval()
    print("Clickbait model loaded.")
except Exception as e:
    print(f"Warning: Could not load Clickbait model: {e}")
    clickbait_tokenizer = None
    clickbait_model = None

print(f"Loading Meta-Classifier from: {META_MODEL_PATH} ...")
try:
    meta_model         = joblib.load(os.path.join(META_MODEL_PATH, "logistic_regression_metaclassifier.joblib"))
    meta_scaler        = joblib.load(os.path.join(META_MODEL_PATH, "meta_feature_scaler.joblib"))
    meta_tfidf         = joblib.load(os.path.join(META_MODEL_PATH, "tfidf_vectorizer.joblib"))
    meta_label_encoder = joblib.load(os.path.join(META_MODEL_PATH, "label_encoder.joblib"))
    print("Meta-Classifier loaded.")
    print(f"  meta_model.n_features_in_ = {getattr(meta_model, 'n_features_in_', 'unknown')}, "
          f"tfidf vocab = {len(getattr(meta_tfidf, 'vocabulary_', {}))}, "
          f"scaler features = {getattr(meta_scaler, 'n_features_in_', 'unknown')}")
except Exception as e:
    print(f"Warning: Could not load Meta-Classifier: {e}")
    meta_model = meta_scaler = meta_tfidf = meta_label_encoder = None

SENTIMENT_LABEL_MAP = {
    0: "negative", 1: "neutral", 2: "positive",
    "LABEL_0": "negative", "LABEL_1": "neutral", "LABEL_2": "positive"
}
CLICKBAIT_LABEL_MAP = {
    0: "Not Clickbait", 1: "Clickbait",
    "LABEL_0": "Not Clickbait", "LABEL_1": "Clickbait"
}
id2label_sentiment = getattr(sentiment_model.config, "id2label", {})

def get_sentiment_label(idx):
    raw = id2label_sentiment.get(idx, id2label_sentiment.get(str(idx), idx))
    return SENTIMENT_LABEL_MAP.get(raw, SENTIMENT_LABEL_MAP.get(idx, str(raw).lower()))

def get_clickbait_label(idx):
    return CLICKBAIT_LABEL_MAP.get(idx, "Clickbait" if idx == 1 else "Not Clickbait")

def article_sentiment_text(article):
    title = article.get("title") or ""
    body  = article.get("body") or article.get("description") or article.get("text") or ""
    if not isinstance(title, str): title = ""
    if not isinstance(body,  str): body  = ""
    body = body[:MAX_SENTIMENT_BODY_CHARS]
    return f"{title}\n{body}" if body else title

def run_model(tokenizer, model, texts):
    """Tokenize, run on DEVICE, return (logits, probs) as CPU tensors."""
    inputs = to_device(tokenizer(texts, padding=True, truncation=True, max_length=512, return_tensors="pt"))
    with torch.no_grad():
        logits = model(**inputs).logits.float().cpu()
    return logits, F.softmax(logits, dim=-1)

TRUTHFULNESS_WEIGHTS = {
    "pants-fire": 0.0, "false": 0.15, "barely-true": 0.35,
    "half-true": 0.65, "mostly-true": 0.85, "true": 1.0
}

_padding_warned = False

def predict_reliability_meta(texts, base_probs_list):
    global _padding_warned
    if not all([meta_model, meta_scaler, meta_tfidf, meta_label_encoder]):
        return None
    try:
        n = len(texts)
        X_num_scaled = meta_scaler.transform(np.array(base_probs_list, dtype=np.float32))
        X_tfidf = meta_tfidf.transform(texts)
        target_dim  = getattr(meta_model, "n_features_in_", 15512)
        current_dim = X_tfidf.shape[1] + X_num_scaled.shape[1]
        remainder   = max(0, target_dim - current_dim)
        if remainder > 0 and not _padding_warned:
            # This padding hides a feature mismatch with training. Check that the
            # training feature order was [tfidf, numeric] and what the extra columns were.
            print(f"WARNING: meta features = {current_dim}, model expects {target_dim}; "
                  f"padding {remainder} zero columns. Scores may be unreliable.")
            _padding_warned = True
        X_full = hstack([X_tfidf, X_num_scaled, csr_matrix((n, remainder))]).tocsr() if remainder > 0 \
                 else hstack([X_tfidf, X_num_scaled]).tocsr()
        preds_proba = meta_model.predict_proba(X_full)
        classes = meta_label_encoder.classes_
        results = []
        for i in range(n):
            probs_i   = preds_proba[i]
            prob_dict = {str(classes[j]): round(float(probs_i[j]), 4) for j in range(len(classes))}
            score     = max(0, min(100, int(round(
                sum(prob_dict.get(c, 0.0) * w for c, w in TRUTHFULNESS_WEIGHTS.items()) * 100))))
            results.append({
                "reliabilityScore": score,
                "truthfulnessLabel": str(classes[int(np.argmax(probs_i))]),
                "metaProbabilities": prob_dict,
                "available": True
            })
        return results
    except Exception as e:
        print(f"Meta-classifier error: {e}", flush=True)
        return None

app = Flask(__name__)
CORS(app, origins=ALLOWED_ORIGINS)

@app.before_request
def require_ml_api_key():
    if ML_API_KEY is None or request.method == "OPTIONS":
        return None
    provided = request.headers.get("X-ML-Key", "")
    if not hmac.compare_digest(provided.encode(), ML_API_KEY.encode()):
        return jsonify({"error": "Unauthorized"}), 401

@app.route("/", methods=["GET"])
def index():
    return jsonify({
        "status": "online",
        "service": "ReliabilityNet Intelligence API (Kaggle)",
        "endpoints": {
            "GET /": "Service info",
            "GET /health": "Health status",
            "POST /predict/sentiment": "Sentiment analysis",
            "POST /predict/clickbait": "Clickbait detection",
            "POST /predict/reliability": "Full reliability pipeline"
        }
    })

@app.route("/health", methods=["GET"])
def health():
    return jsonify({
        "status": "healthy",
        "device": str(DEVICE),
        "clickbait_loaded": clickbait_model is not None,
        "sentiment_loaded": sentiment_model is not None,
        "meta_classifier_loaded": meta_model is not None
    })

@app.route("/predict/sentiment", methods=["POST"])
def predict_sentiment():
    data     = request.get_json(silent=True) or {}
    articles = data.get("articles", [])
    texts    = [article_sentiment_text(a) for a in articles] if articles else data.get("texts", [])
    if not texts:
        return jsonify({"error": "No texts or articles provided"}), 400
    try:
        logits, probs = run_model(sentiment_tokenizer, sentiment_model, texts)
        results = []
        for i, text in enumerate(texts):
            p = probs[i].tolist()
            idx = int(torch.argmax(logits[i]).item())
            results.append({
                "title": articles[i].get("title", text) if articles and i < len(articles) else text,
                "sentiment": get_sentiment_label(idx),
                "confidence": round(p[idx], 4),
                "probabilities": {get_sentiment_label(j): round(p[j], 4) for j in range(len(p))},
                "index": idx
            })
        return jsonify({"success": True, "predictions": results})
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route("/predict/clickbait", methods=["POST"])
def predict_clickbait():
    if not clickbait_model or not clickbait_tokenizer:
        return jsonify({"error": "Clickbait model not loaded"}), 503
    data     = request.get_json(silent=True) or {}
    articles = data.get("articles", [])
    titles   = [a.get("title", "") for a in articles] if articles else data.get("titles", [])
    if not titles:
        return jsonify({"error": "No titles or articles provided"}), 400
    try:
        logits, probs = run_model(clickbait_tokenizer, clickbait_model, titles)
        results = []
        for i in range(len(titles)):
            p   = probs[i].tolist()
            idx = int(torch.argmax(logits[i]).item())
            results.append({
                "title": titles[i],
                "clickbait": get_clickbait_label(idx),
                "confidence": round(p[idx], 4),
                "probabilities": {"Not Clickbait": round(p[0], 4), "Clickbait": round(p[1], 4)}
            })
        return jsonify({"success": True, "predictions": results})
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route("/predict/reliability", methods=["POST"])
def predict_reliability():
    data            = request.get_json(silent=True) or {}
    articles        = data.get("articles", [])
    titles          = [a.get("title") or "" for a in articles] if articles else data.get("texts", [])
    sentiment_texts = [article_sentiment_text(a) for a in articles] if articles else titles
    if not titles:
        return jsonify({"error": "No texts or articles provided"}), 400
    try:
        s_logits, s_probs = run_model(sentiment_tokenizer, sentiment_model, sentiment_texts)
        cb_probs_list = []
        if clickbait_model and clickbait_tokenizer:
            try:
                _, cb_probs = run_model(clickbait_tokenizer, clickbait_model, titles)
                cb_probs_list = cb_probs.tolist()
            except Exception as e:
                print(f"Clickbait error: {e}", flush=True)
        base_probs_for_meta = []
        for i in range(len(titles)):
            p = s_probs[i].tolist()
            cb_p = cb_probs_list[i][1] if i < len(cb_probs_list) and len(cb_probs_list[i]) > 1 else 0.5
            # NOTE: the first feature is a fixed 0.5 placeholder, as in the original
            # notebook. Check what this feature was when the meta-classifier was trained.
            base_probs_for_meta.append([0.5, cb_p, p[0] if len(p) > 0 else 0.33,
                                                   p[1] if len(p) > 1 else 0.33,
                                                   p[2] if len(p) > 2 else 0.33])
        meta_results = predict_reliability_meta(titles, base_probs_for_meta)
        results = []
        for i, text in enumerate(titles):
            p     = s_probs[i].tolist()
            s_idx = int(torch.argmax(s_logits[i]).item())
            sentiment_result = {
                "label": get_sentiment_label(s_idx),
                "confidence": round(p[s_idx], 4),
                "probabilities": {get_sentiment_label(j): round(p[j], 4) for j in range(len(p))}
            }
            clickbait_result = None
            if i < len(cb_probs_list):
                cb_p    = cb_probs_list[i]
                cb_idx  = int(np.argmax(cb_p))
                clickbait_result = {
                    "label": get_clickbait_label(cb_idx),
                    "confidence": round(cb_p[cb_idx], 4),
                    "probabilities": {"Not Clickbait": round(cb_p[0], 4), "Clickbait": round(cb_p[1], 4)}
                }
            meta_pred = (meta_results[i] if meta_results and i < len(meta_results)
                         else {"reliabilityScore": None, "truthfulnessLabel": None, "metaProbabilities": None, "available": False})
            article_text = (articles[i].get("text") or articles[i].get("body") or
                            articles[i].get("description") or text) if articles and i < len(articles) else text
            results.append({
                "uuid":  articles[i].get("uuid")  if articles and i < len(articles) else None,
                "title": articles[i].get("title", text) if articles and i < len(articles) else text,
                "text":  article_text,
                "sentiment":   sentiment_result,
                "clickbait":   clickbait_result,
                "reliability": {
                    "reliabilityScore":   meta_pred["reliabilityScore"],
                    "truthfulnessLabel":  meta_pred["truthfulnessLabel"],
                    "probabilities":      meta_pred["metaProbabilities"],
                    "available":          meta_pred["available"]
                }
            })
        return jsonify({"success": True, "predictions": results})
    except Exception as e:
        return jsonify({"error": str(e)}), 500

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=PORT, debug=False)
'''

with open("/kaggle/working/model_server.py", "w") as f:
    f.write(MODEL_SERVER_CODE)

print('✅ model_server.py written to /kaggle/working/')

In [ ]:
# ── 6. Start Flask as a background process (with a log file) ─────────────────
import subprocess
import sys
import time
import requests as req_lib

FLASK_PORT = 5001
LOG_PATH   = "/kaggle/working/flask.log"

# Stop any server left over from an earlier run of this cell
try:
    if flask_proc.poll() is None:
        flask_proc.terminate()
        flask_proc.wait(timeout=15)
except NameError:
    pass
subprocess.run(["pkill", "-f", "model_server.py"], check=False)
time.sleep(1)

def show_flask_log(n=40):
    """Print the last n lines of the Flask server log."""
    with open(LOG_PATH) as f:
        print("".join(f.readlines()[-n:]))

log_file = open(LOG_PATH, "w")
flask_proc = subprocess.Popen(
    [sys.executable, "-u", "/kaggle/working/model_server.py"],
    env={**os.environ, "PORT": str(FLASK_PORT)},
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

print("Waiting for Flask to start (model loading can take a few minutes)", end="")
for _ in range(120):  # up to 10 minutes
    if flask_proc.poll() is not None:
        print(f"\n❌ Flask exited with code {flask_proc.returncode}. Log:\n")
        show_flask_log()
        raise RuntimeError("Flask server failed to start — see the log above.")
    try:
        r = req_lib.get(f"http://127.0.0.1:{FLASK_PORT}/health", timeout=3)
        if r.status_code == 200:
            print(" ✅ Flask is up!")
            print(r.json())
            break
    except req_lib.RequestException:
        pass
    time.sleep(5)
    print(".", end="", flush=True)
else:
    print("\n❌ Flask did not respond within 10 minutes. Last log lines:\n")
    show_flask_log()
    raise RuntimeError("Flask server did not start in time.")

show_flask_log(15)

In [ ]:
# ── 7. Expose Flask publicly via ngrok (fixed domain) ─────────────────────────
# The API gets the SAME public URL every run (Esther's ngrok dev domain).
# Requires a Kaggle Secret named NGROK_TOKEN (Add-ons → Secrets, ticked for this
# notebook) from the ngrok account that owns the domain.
# A domain can only be online in ONE session at a time.
from pyngrok import ngrok, conf

NGROK_DOMAIN = "jockstrap-track-paradox.ngrok-free.dev"  # dashboard.ngrok.com → Domains

FLASK_PUBLIC_URL = None
NGROK_AUTH_TOKEN = get_secret("NGROK_TOKEN")

if not NGROK_AUTH_TOKEN:
    print("⚠️  NGROK_TOKEN not found — skipping the public API tunnel.")
    print("    Add it under Add-ons → Secrets (name: NGROK_TOKEN) and re-run this cell.")
else:
    conf.get_default().auth_token = NGROK_AUTH_TOKEN  # no config-file write, safe to re-run
    ngrok.kill()  # close tunnels from earlier runs
    try:
        # ngrok's current endpoint format takes the address as `url` (not `domain`)
        tunnel = ngrok.connect(FLASK_PORT, url=f"https://{NGROK_DOMAIN}")
    except Exception as e:
        print(f"ℹ️  url= not accepted ({e}); using the account's dev domain instead.")
        ngrok.kill()
        tunnel = ngrok.connect(FLASK_PORT)  # free accounts land on their dev domain
    FLASK_PUBLIC_URL = tunnel.public_url
    print(f"🌍 Public Flask API URL: {FLASK_PUBLIC_URL}")
    print(f"   Health check: {FLASK_PUBLIC_URL}/health")
    if NGROK_DOMAIN not in FLASK_PUBLIC_URL:
        print(f"⚠️  Expected {NGROK_DOMAIN} — check the token belongs to the account that owns it.")

In [ ]:
# ── 8. Gradio UI ──────────────────────────────────────────────────────────────
import json
import gradio as gr
import requests

API_BASE = f"http://127.0.0.1:{FLASK_PORT}"  # Gradio calls Flask locally for speed
PUBLIC_URL_TEXT = FLASK_PUBLIC_URL or "not available (ngrok not configured)"

API_KEY = os.environ.get("ML_API_KEY", "")
HEADERS = {"Content-Type": "application/json"}
if API_KEY:
    HEADERS["X-ML-Key"] = API_KEY

try:
    DEVICE_TEXT = req_lib.get(f"{API_BASE}/health", timeout=5).json().get("device", "unknown")
except Exception:
    DEVICE_TEXT = "unknown"


def call_api(path, payload, timeout=120):
    """POST to the Flask API and surface the server's own error message."""
    r = requests.post(f"{API_BASE}{path}", json=payload, headers=HEADERS, timeout=timeout)
    try:
        data = r.json()
    except ValueError:
        data = {"error": r.text[:500]}
    if r.status_code != 200 or "error" in data:
        raise RuntimeError(f"HTTP {r.status_code}: {data.get('error', data)}")
    return data


def bar(p, width=20):
    return "█" * int((p or 0) * width)


# ── Tab 1 – Sentiment ─────────────────────────────────────────────────────────
def predict_sentiment(title, body):
    if not (title or "").strip() and not (body or "").strip():
        return "⚠️ Please enter a title or body."
    try:
        pred  = call_api("/predict/sentiment", {"articles": [{"title": title, "body": body}]}, 60)["predictions"][0]
        label = pred["sentiment"].upper()
        md  = f"### 🎭 Sentiment: **{label}** ({pred['confidence']*100:.1f}% confidence)\n\n"
        md += "| Label | Probability |\n|---|---|\n"
        for lbl, p in pred["probabilities"].items():
            md += f"| {lbl} | {bar(p)} {p:.4f} |\n"
        return md
    except Exception as e:
        return f"❌ Error: {e}"


# ── Tab 2 – Clickbait ─────────────────────────────────────────────────────────
def predict_clickbait(title):
    if not (title or "").strip():
        return "⚠️ Please enter a headline."
    try:
        pred  = call_api("/predict/clickbait", {"titles": [title]}, 60)["predictions"][0]
        label = pred["clickbait"]
        icon  = "🚨" if label == "Clickbait" else "✅"
        md  = f"### {icon} Clickbait: **{label}** ({pred['confidence']*100:.1f}% confidence)\n\n"
        md += "| Label | Probability |\n|---|---|\n"
        for lbl, p in pred["probabilities"].items():
            md += f"| {lbl} | {bar(p)} {p:.4f} |\n"
        return md
    except Exception as e:
        return f"❌ Error: {e}"


# ── Tab 3 – Full Reliability Pipeline ─────────────────────────────────────────
def predict_reliability(title, body):
    if not (title or "").strip():
        return "⚠️ Please enter a title."
    try:
        pred = call_api("/predict/reliability", {"articles": [{"title": title, "body": body}]}, 120)["predictions"][0]

        rel         = pred.get("reliability") or {}
        score       = rel.get("reliabilityScore")
        truth_label = rel.get("truthfulnessLabel") or "N/A"
        if score is None:
            score_line = "### 📊 Reliability Score: **N/A** (meta-classifier unavailable — check `show_flask_log()`)"
        else:
            filled = int(score // 5)
            score_line = f"### 📊 Reliability Score: **{score}/100**\n`{'█' * filled}{'░' * (20 - filled)}` {score}%"

        sent   = pred.get("sentiment") or {}
        s_lbl  = (sent.get("label") or "N/A").upper()
        s_conf = sent.get("confidence") or 0

        cb      = pred.get("clickbait") or {}
        cb_lbl  = cb.get("label") or "N/A"
        cb_conf = cb.get("confidence") or 0
        cb_icon = {"Clickbait": "🚨", "Not Clickbait": "✅"}.get(cb_lbl, "⚠️")

        md  = "## 🧠 Reliability Report\n\n"
        md += score_line + "\n\n"
        md += f"**Truthfulness Label:** `{truth_label}`\n\n---\n\n"
        md += f"### 🎭 Sentiment: **{s_lbl}** ({s_conf*100:.1f}%)\n"
        for lbl, p in (sent.get("probabilities") or {}).items():
            md += f"- {lbl}: {bar(p)} {p:.4f}\n"

        if cb:
            md += f"\n### {cb_icon} Clickbait: **{cb_lbl}** ({cb_conf*100:.1f}%)\n"
            for lbl, p in (cb.get("probabilities") or {}).items():
                md += f"- {lbl}: {bar(p)} {p:.4f}\n"
        else:
            md += "\n### ⚠️ Clickbait: model not loaded\n"

        if rel.get("probabilities"):
            md += "\n### 📋 Truthfulness Probabilities\n| Label | Probability |\n|---|---|\n"
            for lbl, p in sorted(rel["probabilities"].items(), key=lambda x: -x[1]):
                md += f"| {lbl} | {bar(p)} {p:.4f} |\n"
        return md
    except Exception as e:
        return f"❌ Error: {e}"


# ── Tab 4 – Batch / Raw JSON ──────────────────────────────────────────────────
def batch_reliability(json_input):
    try:
        payload = json.loads(json_input)
        return json.dumps(call_api("/predict/reliability", payload, 180), indent=2)
    except Exception as e:
        return json.dumps({"error": str(e)}, indent=2)


# ── Build the Gradio app ──────────────────────────────────────────────────────
EXAMPLE_TITLE = "10 Shocking Secrets Doctors Don't Want You to Know!"
EXAMPLE_BODY  = "A new study from Harvard reveals that common medications may have hidden side effects."
EXAMPLE_BATCH = json.dumps({
    "articles": [
        {"title": "Scientists discover water on Mars", "body": "NASA confirms the finding."},
        {"title": "You won't BELIEVE what happens next!", "body": "Click here to find out."}
    ]
}, indent=2)

CURL_BASE = FLASK_PUBLIC_URL or API_BASE
CURL_EXAMPLE = (
    f"curl -X POST {CURL_BASE}/predict/reliability \\\n"
    "  -H 'Content-Type: application/json' \\\n"
    + (f"  -H 'X-ML-Key: <your key>' \\\n" if API_KEY else "")
    + """  -d '{"articles": [{"title": "Scientists discover life on Mars", "body": "NASA confirms."}]}'"""
)

# Gradio 6 moved `theme` from gr.Blocks() to launch()
GRADIO_MAJOR  = int(gr.__version__.split(".")[0])
THEME         = gr.themes.Soft()
blocks_kwargs = {} if GRADIO_MAJOR >= 6 else {"theme": THEME}
launch_kwargs = {"theme": THEME} if GRADIO_MAJOR >= 6 else {}

gr.close_all()  # close UIs from earlier runs of this cell

with gr.Blocks(title="ReliabilityNet", **blocks_kwargs) as demo:
    gr.Markdown(
        "# 🧠 ReliabilityNet – Article Analysis\n"
        "Powered by RoBERTa sentiment, RoBERTa-large clickbait, and a Logistic Regression meta-classifier.  \n"
        f"Flask API on Kaggle (device: **{DEVICE_TEXT}**) · Public API URL: **{PUBLIC_URL_TEXT}**"
    )

    with gr.Tab("🎭 Sentiment"):
        gr.Markdown("Classify the emotional tone of an article (negative / neutral / positive).")
        s_title = gr.Textbox(label="Article Title", value=EXAMPLE_TITLE, lines=1)
        s_body  = gr.Textbox(label="Article Body (optional, up to 500 chars used)", value=EXAMPLE_BODY, lines=4)
        s_btn   = gr.Button("Analyse Sentiment", variant="primary")
        s_out   = gr.Markdown()
        s_btn.click(predict_sentiment, inputs=[s_title, s_body], outputs=s_out)

    with gr.Tab("🚨 Clickbait"):
        gr.Markdown("Detect whether a headline is clickbait.")
        cb_title = gr.Textbox(label="Article Title", value=EXAMPLE_TITLE, lines=1)
        cb_btn   = gr.Button("Check Clickbait", variant="primary")
        cb_out   = gr.Markdown()
        cb_btn.click(predict_clickbait, inputs=[cb_title], outputs=cb_out)

    with gr.Tab("📊 Full Reliability"):
        gr.Markdown("Run the full pipeline: sentiment + clickbait + meta-classifier reliability score.")
        r_title = gr.Textbox(label="Article Title", value=EXAMPLE_TITLE, lines=1)
        r_body  = gr.Textbox(label="Article Body (optional)", value=EXAMPLE_BODY, lines=4)
        r_btn   = gr.Button("Analyse Reliability", variant="primary")
        r_out   = gr.Markdown()
        r_btn.click(predict_reliability, inputs=[r_title, r_body], outputs=r_out)

    with gr.Tab("📦 Batch / Raw JSON"):
        gr.Markdown(
            "Send a raw JSON payload to `/predict/reliability`.  \n"
            'Format: `{"articles": [{"title": "...", "body": "..."}]}`'
        )
        b_in  = gr.Code(label="Request JSON", value=EXAMPLE_BATCH, language="json", lines=12)
        b_btn = gr.Button("Submit", variant="primary")
        b_out = gr.Code(label="Response JSON", language="json", lines=20)
        b_btn.click(batch_reliability, inputs=[b_in], outputs=b_out)

    with gr.Tab("🔗 API Info"):
        gr.Markdown(
            "## Endpoints\n\n"
            "| Method | Path | Description |\n|--------|------|-------------|\n"
            "| GET  | `/` | Service info |\n"
            "| GET  | `/health` | Model health check |\n"
            "| POST | `/predict/sentiment` | Sentiment (title + body) |\n"
            "| POST | `/predict/clickbait` | Clickbait (title only) |\n"
            "| POST | `/predict/reliability` | Full pipeline |\n\n"
            f"**Public base URL:** `{PUBLIC_URL_TEXT}`\n\n"
            "### Example curl\n```bash\n" + CURL_EXAMPLE + "\n```\n"
        )

print("\n🚀 Launching Gradio UI...")
demo.launch(share=True, quiet=True, **launch_kwargs)
print("🌍 Gradio public link:", demo.share_url)

## 🛑 Teardown
Run the cell below to stop Gradio, the ngrok tunnel and the Flask server.


In [ ]:
# Run this cell to cleanly shut everything down
try:
    demo.close()
except Exception:
    pass
try:
    ngrok.kill()
except Exception:
    pass
try:
    flask_proc.terminate()
except Exception:
    pass
print("Gradio, ngrok tunnel and Flask server stopped.")